# Arrays and tensors

The core idea of idris-ml: **tensor shapes are part of the type**. A `Vector 3 Double`
is a different type from `Vector 4 Double`, so shape mismatches become compile errors
instead of runtime crashes.

In PyTorch, `torch.tensor([1,2,3])` has shape `(3,)` as a runtime property.
In idris-ml, the shape is in the type itself, and the compiler checks it.

Two related types: `Array [dims] ty` is pure-Idris structural data (Vect-of-Vect with rank-N Functor/Num/Floating instances), and `Tensor dims ex dt g` is the autograd-tracked tensor on a backend. This tutorial focuses on `Array`; tutorial 02 introduces `Tensor`.


## The Array type

The structural data type is built from one GADT, indexed by a list of dimension sizes:


In [1]:
:t Array


Tensor.Tensor : Vect rank Nat -> Type -> Type


The first argument `Vect rank Nat` is a compile-time list of dimension sizes.
The second is the element type. Three aliases cover common ranks:


In [2]:
:t Scalar


0 Tensor.Scalar : Type -> Type


In [3]:
:t Vector


0 Tensor.Vector : Nat -> Type -> Type


In [4]:
:t Matrix


0 Tensor.Matrix : Nat -> Nat -> Type -> Type


## Creating arrays

`SArray` wraps a scalar value. `VArray` collects sub-arrays into the next rank.
Numeric literals auto-convert via `Num` and `FromDouble` instances.


In [5]:
the (Scalar Double) (SArray 3.14)


STensor 3.14


In [6]:
the (Vector 3 Double) (VArray [1.0, 2.0, 3.0])


VTensor [STensor 1.0, STensor 2.0, STensor 3.0]


In [7]:
the (Matrix 2 3 Double) (VArray [VArray [1, 2, 3], VArray [4, 5, 6]])


VTensor [VTensor [STensor 1.0, STensor 2.0, STensor 3.0], VTensor [STensor 4.0, STensor 5.0, STensor 6.0]]


## Arithmetic

Standard operators work elementwise on arrays of the same shape.

**Important:** `(*)` is elementwise multiplication, not matrix multiply.
Use `matrixMultiply` or `matrixVectorMultiply` for linear algebra.


In [8]:
the (Vector 3 Double) (VArray [1, 2, 3]) + the (Vector 3 Double) (VArray [10, 20, 30])


VTensor [STensor 11.0, STensor 22.0, STensor 33.0]


In [9]:
the (Vector 3 Double) (VArray [2, 3, 4]) * the (Vector 3 Double) (VArray [10, 10, 10])


VTensor [STensor 20.0, STensor 30.0, STensor 40.0]


## Shape safety

Vectors of different sizes are different types, so they can't be combined:


In [10]:
the (Vector 3 Double) (VArray [1, 2, 3]) + the (Vector 2 Double) (VArray [1, 2])


Error: When unifying:
    Tensor [2] Double
and:
    Tensor [3] Double
Mismatch between: 0 and 1.

(Interactive):1:45--1:83
 1 | the (Vector 3 Double) (VTensor [1, 2, 3]) + the (Vector 2 Double) (VTensor [1, 2])
                                                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


In PyTorch, mismatched shapes are caught at runtime. Here the compiler catches
them before any code runs.


## Linear algebra

Matrix operations carry dimension constraints in their types:


In [11]:
:t matrixVectorMultiply


Math.matrixVectorMultiply : Num ty => Matrix m n ty -> Vector n ty -> Vector m ty


The inner dimensions must match: a `Matrix m n` can only multiply a `Vector n`,
producing a `Vector m`. Passing the wrong size is a type error.


In [12]:
:t matrixMultiply


Math.matrixMultiply : Num ty => Matrix m n ty -> Matrix n k ty -> Matrix m k ty


In [13]:
:t dotProduct


Math.dotProduct : Num ty => Vector n ty -> Vector n ty -> ty


## Transforming arrays

Apply functions elementwise with `map`:


In [14]:
map (*2) (the (Vector 4 Double) (VArray [1, 2, 3, 4]))


VTensor [STensor 2.0, STensor 4.0, STensor 6.0, STensor 8.0]


In [15]:
dotProduct (the (Vector 3 Double) (VArray [1, 2, 3])) (the (Vector 3 Double) (VArray [4, 5, 6]))


32.0


## Discovering the API

Use `:t` to check any function's type, `:doc` for documentation, and `:browse` to
list everything in a module. The queries read the installed library directly, so
this reference cannot drift from the code.


In [16]:
:t softmax


Math.softmax : (Fractional ty, Floating ty) => NormalizationFunction ty


In [17]:
:t crossEntropy


Math.crossEntropy : (Num ty, (Neg ty, (Floating ty, (Fractional ty, Ord ty)))) => LossFunction ty


In [18]:
:doc dotProduct


Math.dotProduct : Num ty => Vector n ty -> Vector n ty -> ty
  Visibility: export


Next: [02 Building models](02_building_models.ipynb) — composing layers into networks
with compile-time dimension checking.
